# Titanic
## Score:.75598

In [5]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import loguniform, randint, uniform
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = Path.cwd()
DATA = ROOT / "titanic"
assert (DATA / "train.csv").exists(), "Set cwd to project root (folder containing titanic/train.csv)"

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [6]:
def extract_title(name: str) -> str:
    m = re.search(r",\s*([^\.]+)\.", name)
    return m.group(1).strip() if m else "Unknown"


def cabin_deck(cabin) -> str:
    if pd.isna(cabin) or not str(cabin).strip():
        return "U"
    c = str(cabin).strip()[0]
    return c if c.isalpha() else "U"


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["Title"] = out["Name"].map(extract_title)
    rare = {
        "Lady",
        "Countess",
        "Capt",
        "Col",
        "Don",
        "Dr",
        "Major",
        "Rev",
        "Sir",
        "Jonkheer",
        "Dona",
    }
    out["Title"] = out["Title"].replace(
        {
            "Mlle": "Miss",
            "Ms": "Miss",
            "Mme": "Mrs",
        }
    )
    out.loc[out["Title"].isin(rare), "Title"] = "Rare"
    out["FamilySize"] = out["SibSp"] + out["Parch"] + 1
    out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
    out["HasCabin"] = out["Cabin"].notna().astype(int)
    out["Deck"] = out["Cabin"].map(cabin_deck)
    out["FarePerPerson"] = out["Fare"] / out["FamilySize"].clip(lower=1)
    out["LogFare"] = np.log1p(out["Fare"].fillna(0))
    out["IsChild"] = np.where(out["Age"].notna(), (out["Age"] < 16).astype(int), 0)
    out["Pclass_Sex"] = out["Pclass"].astype(str) + "_" + out["Sex"].astype(str)
    out["Title_Pclass"] = out["Title"].astype(str) + "_" + out["Pclass"].astype(str)
    return out

In [7]:
def build_pipeline(**model_kw) -> Pipeline:
    numeric = [
        "Pclass",
        "Age",
        "SibSp",
        "Parch",
        "Fare",
        "FamilySize",
        "IsAlone",
        "HasCabin",
        "FarePerPerson",
        "LogFare",
        "IsChild",
    ]
    categorical = ["Sex", "Embarked", "Title", "Deck", "Pclass_Sex", "Title_Pclass"]

    pre = ColumnTransformer(
        [
            (
                "num",
                IterativeImputer(random_state=42, max_iter=25),
                numeric,
            ),
            (
                "cat",
                Pipeline(
                    [
                        ("impute", SimpleImputer(strategy="most_frequent")),
                        ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                    ]
                ),
                categorical,
            ),
        ]
    )

    defaults = dict(
        random_state=42,
        max_iter=250,
        learning_rate=0.06,
        max_depth=6,
        min_samples_leaf=15,
        l2_regularization=0.1,
        early_stopping=True,
        validation_fraction=0.12,
        n_iter_no_change=15,
    )
    defaults.update(model_kw)
    clf = HistGradientBoostingClassifier(**defaults)
    return Pipeline([("prep", pre), ("model", clf)])

In [8]:
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")

train_x = add_features(train.drop(columns=["Survived"]))
train_y = train["Survived"]
test_x = add_features(test)

base = build_pipeline()
base_cv = cross_val_score(base, train_x, train_y, cv=CV, scoring="accuracy", n_jobs=-1)
print(f"Default HGB (CV accuracy): {base_cv.mean():.4f} +/- {base_cv.std():.4f}")

param_dist = {
    "model__learning_rate": uniform(0.03, 0.17),
    "model__max_iter": randint(150, 450),
    "model__max_depth": randint(3, 12),
    "model__min_samples_leaf": randint(5, 35),
    "model__l2_regularization": loguniform(1e-4, 0.5),
    "model__max_leaf_nodes": randint(20, 64),
}

search = RandomizedSearchCV(
    build_pipeline(),
    param_distributions=param_dist,
    n_iter=35,
    cv=CV,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
search.fit(train_x, train_y)
print("Best CV accuracy:", round(search.best_score_, 5))
print("Best params:", search.best_params_)

best = search.best_estimator_
pred = best.predict(test_x)
sub = pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": pred.astype(int)})
out_path = ROOT / "submission.csv"
sub.to_csv(out_path, index=False)
print(f"Wrote {out_path} ({len(sub)} rows)")
sub.head(10)

Default HGB (CV accuracy): 0.8428 +/- 0.0157
Fitting 5 folds for each of 35 candidates, totalling 175 fits
Best CV accuracy: 0.84509
Best params: {'model__l2_regularization': np.float64(0.000769364685182322), 'model__learning_rate': np.float64(0.05463212825550792), 'model__max_depth': 8, 'model__max_iter': 396, 'model__max_leaf_nodes': 23, 'model__min_samples_leaf': 27}
Wrote c:\Users\ol1v3_7dwns5u\OneDrive\Desktop\ACTIVE\Titanic\submission.csv (418 rows)


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
5,897,0
6,898,0
7,899,0
8,900,1
9,901,0
